# Analyse du dataset SGDBE INRAE — Carte des sols de France

## Source
**Base de Données Géographique des Sols de France (SGDBE)**  
Produite par : **INRAE** (Institut national de recherche pour l'agriculture, l'alimentation et l'environnement)  
Téléchargée sur : **data.gouv.fr** — référence **30169**  
Version : 1.0  
Licence : Etalab (open data)

## Particularité de ce dataset
Contrairement au RPG qui est un fichier par parcelle, le SGDBE est une **carte vectorielle de polygones de sol**.  
Chaque polygone couvre une zone géographique et lui associe un type de sol.  
La résolution est **1/1 000 000** — un polygone couvre en moyenne ~25 km² en HdF.

## Structure du dataset
Le SGDBE se compose de 3 fichiers complémentaires :
- `30169_L93.zip` → les **polygones géographiques** (Shapefile en Lambert 93)
- `stu.tab` → les **propriétés de chaque type de sol** (texture, drainage, pente...)
- `stuorg.tab` → la **relation** entre polygones et types de sol

## Ce qu'on fait
1. Lire et explorer chaque fichier brut
2. Comprendre le système SMU/STU
3. Extraire le type de sol dominant par polygone
4. Convertir les coordonnées Lambert 93 → WGS84
5. Joindre spatialement chaque parcelle RPG au polygone de sol le plus proche

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — Imports                                     ║
# ╚══════════════════════════════════════════════════════════╝

!pip install scipy matplotlib pandas numpy -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import struct, math, warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = '#fafafa'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

COULEURS_SOL = {
    'limoneux':          '#f59e0b',
    'argilo_limoneux':   '#d97706',
    'sablo_limoneux':    '#fbbf24',
    'argileux_lourd':    '#92400e',
    'tourbe':            '#166534',
    'craie':             '#e5e7eb',
}
print('✓ Imports OK')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — Upload des fichiers                         ║
# ╚══════════════════════════════════════════════════════════╝

from google.colab import files
print('Uploader les 4 fichiers SGDBE :')
print('  1. 30169_L93.zip  (polygones géographiques)')
print('  2. stu.tab        (propriétés des types de sol)')
print('  3. stuorg.tab     (relation polygones ↔ types de sol)')
print('  4. rpg_coords.csv (coordonnées des parcelles RPG — issu notebook RPG 2024)')
uploaded = files.upload()
print(f'\n✓ Fichiers reçus : {list(uploaded.keys())}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — Exploration du Shapefile (30169_L93.zip)    ║
# ║                                                          ║
# ║  Le zip contient un Shapefile en Lambert 93.             ║
# ║  Un Shapefile est un ensemble de fichiers :              ║
# ║  .shp = géométries | .dbf = attributs | .prj = proj.     ║
# ║                                                          ║
# ║  On lit le .dbf sans geopandas (plus léger) pour         ║
# ║  explorer les attributs des polygones de sol.            ║
# ╚══════════════════════════════════════════════════════════╝

import zipfile, os

# Extraire le zip
with zipfile.ZipFile('30169_L93.zip', 'r') as z:
    z.extractall('sols_france/')
print('Fichiers extraits :', os.listdir('sols_france/'))
print()

# Lire le .dbf manuellement
def read_dbf(path):
    with open(path, 'rb') as f:
        f.read(4)
        n_records    = struct.unpack('<I', f.read(4))[0]
        header_size  = struct.unpack('<H', f.read(2))[0]
        f.read(22)
        fields = []
        while True:
            t = f.read(1)
            if t == b'\r': break
            name = (t + f.read(10)).rstrip(b'\x00').decode('latin-1')
            ftype = f.read(1).decode('latin-1')
            f.read(4)
            flen = struct.unpack('B', f.read(1))[0]
            f.read(15)
            fields.append((name, ftype, flen))
        f.seek(header_size)
        records = []
        for _ in range(n_records):
            deleted = f.read(1)
            row = {n: f.read(l).decode('latin-1').strip() for n,t,l in fields}
            if deleted != b'*':
                records.append(row)
    return pd.DataFrame(records)

dbf = read_dbf('sols_france/30169_L93.dbf')
dbf['SMU'] = pd.to_numeric(dbf['SMU'], errors='coerce')
dbf['AREA'] = pd.to_numeric(dbf['AREA'], errors='coerce')

print('=== ATTRIBUTS DU SHAPEFILE (30169_L93.dbf) ===')
print(f'Nombre de polygones de sol : {len(dbf):,}')
print(f'Nombre de SMU uniques      : {dbf.SMU.nunique()}')
print()
print('Colonnes disponibles :')
for col in dbf.columns:
    print(f'  {col:<15} type={str(dbf[col].dtype):<12}  ex: {dbf[col].iloc[0]}')
print()
print('Explication des colonnes :')
print('  AREA    : surface du polygone en m² (Lambert 93)')
print('  PERIMETER: périmètre du polygone en m')
print('  SOIL_   : numéro interne de l\'entité')
print('  SOIL_ID : identifiant unique du polygone')
print('  SMU     : Soil Mapping Unit — code de l\'Unité Cartographique de Sol')
print('            → clé de jointure avec stu.tab et stuorg.tab')
print()
print(f'Surface totale couverte : {dbf.AREA.sum()/1e9:.0f} km²')
print(f'Surface moyenne par polygone : {dbf.AREA.mean()/1e6:.0f} km²')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 4 — Exploration de stu.tab                      ║
# ║                                                          ║
# ║  STU = Soil Typological Unit                             ║
# ║  C'est la table qui contient les PROPRIÉTÉS de chaque   ║
# ║  type de sol : texture, drainage, pente, altitude...     ║
# ║                                                          ║
# ║  917 unités typologiques pour toute la France.           ║
# ╚══════════════════════════════════════════════════════════╝

def parse_tab(path):
    with open(path, encoding='latin-1') as f:
        lines = f.readlines()
    header = lines[0].strip().split(';')
    records = []
    for line in lines[1:]:
        line = line.strip().strip('"').replace('\\"','').replace('\\','')
        vals = line.split(';')
        if len(vals) >= len(header):
            records.append(dict(zip(header, vals[:len(header)])))
    return pd.DataFrame(records)

stu = parse_tab('stu.tab')
for col in ['stu','text1','text2','slope1','aglim1','dt','roo']:
    if col in stu.columns:
        stu[col] = pd.to_numeric(stu[col], errors='coerce')

print('=== STU.TAB — Propriétés des types de sol ===')
print(f'Nombre d\'unités typologiques (STU) : {len(stu)}')
print()
print('Colonnes disponibles :')

EXPLICATIONS_STU = {
    'stu':    'Identifiant STU — clé de jointure avec stuorg.tab',
    'text1':  'Code texture dominante (1=sablo-lim, 2=limoneux, 3=argilo-lim, 4=argileux, 9=tourbeux)',
    'text2':  'Code texture secondaire',
    'slope1': 'Code de pente dominante (1=0-2%, 2=2-5%, 3=5-15%, 4=>15%)',
    'aglim1': 'Contrainte agricole principale (1=aucune, 2=excès eau, 3=sécheresse...)',
    'dt':     'Type de drainage (1=bien drainé, 5=très mal drainé)',
    'roo':    'Profondeur d\'enracinement (1=très profond, 4=superficiel)',
    'zmin':   'Altitude minimale (m)',
    'zmax':   'Altitude maximale (m)',
    'use1':   'Utilisation principale des terres',
    'soil':   'Code sol européen',
}
for col in stu.columns:
    expl = EXPLICATIONS_STU.get(col, '—')
    print(f'  {col:<12} {expl}')

print()
print('Distribution des textures (text1) — code → type de sol :')
TEXTURE_MAP = {1:'sablo-limoneux', 2:'limoneux', 3:'argilo-limoneux',
               4:'argileux lourd', 5:'très argileux', 9:'tourbeux', 0:'non défini'}
dist_text = stu['text1'].value_counts().sort_index()
for code, n in dist_text.items():
    nom = TEXTURE_MAP.get(int(code) if not np.isnan(code) else 0, '—')
    print(f'  text1={int(code) if not np.isnan(code) else "NaN":<4} → {nom:<20} : {n} STU')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 5 — Exploration de stuorg.tab                   ║
# ║                                                          ║
# ║  STUORG = relation entre SMU (polygones) et STU (types). ║
# ║  Un polygone peut contenir PLUSIEURS types de sol,       ║
# ║  chacun avec un pourcentage de surface (pcarea).         ║
# ║                                                          ║
# ║  Ex : SMU 320053 contient :                              ║
# ║    STU 320199 → 70% de la surface                        ║
# ║    STU 320200 → 20% de la surface                        ║
# ║    STU 320201 → 5%  de la surface                        ║
# ║    STU 320202 → 5%  de la surface                        ║
# ║                                                          ║
# ║  On prend la STU dominante (pcarea max) pour chaque SMU. ║
# ╚══════════════════════════════════════════════════════════╝

stuorg = parse_tab('stuorg.tab')
for col in ['smu','stu','pcarea']:
    stuorg[col] = pd.to_numeric(stuorg[col], errors='coerce')

print('=== STUORG.TAB — Relation SMU ↔ STU ===')
print(f'Nombre de lignes : {len(stuorg)}')
print(f'Colonnes         : {stuorg.columns.tolist()}')
print()
print('Explication des colonnes :')
print('  smu    : identifiant du polygone cartographique (Soil Mapping Unit)')
print('  stu    : identifiant du type de sol (Soil Typological Unit)')
print('  pcarea : % de surface du polygone occupé par ce type de sol')
print()
print('Exemple — SMU 320053 :')
ex = stuorg[stuorg.smu == 320053]
if len(ex) > 0:
    print(ex.to_string())
else:
    print(stuorg.head(8).to_string())
print()

# Nombre moyen de STU par SMU
n_stu_par_smu = stuorg.groupby('smu').size()
print(f'Nombre moyen de types de sol par polygone : {n_stu_par_smu.mean():.1f}')
print(f'Maximum                                   : {n_stu_par_smu.max()}')
print(f'Minimum                                   : {n_stu_par_smu.min()}')
print()
print('→ On prend la STU avec pcarea maximum = type de sol DOMINANT par polygone')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 6 — Construction SMU → propriétés sol           ║
# ║                                                          ║
# ║  On assemble les 3 fichiers pour obtenir une table       ║
# ║  SMU → sol_type + argile + pH + MO                       ║
# ║                                                          ║
# ║  IMPORTANT : argile, pH et MO sont des ESTIMATIONS       ║
# ║  issues de valeurs typiques par type de texture.         ║
# ║  Ce ne sont pas des mesures analytiques réelles.         ║
# ╚══════════════════════════════════════════════════════════╝

# Mappings texture → valeurs agronomiques
# Source : valeurs typiques BDAT INRAE / Chambres Agriculture HdF
TEXTURE_TO_SOL    = {1:'sablo_limoneux', 2:'limoneux', 3:'argilo_limoneux',
                     4:'argileux_lourd', 5:'argileux_lourd', 9:'tourbe', 0:'limoneux'}
TEXTURE_TO_ARGILE = {1:8,  2:18, 3:28, 4:42, 5:52, 9:15, 0:20}
TEXTURE_TO_PH     = {1:6.2, 2:6.8, 3:6.6, 4:6.4, 5:6.3, 9:5.8, 0:6.7}
TEXTURE_TO_MO     = {1:1.5, 2:2.5, 3:2.8, 4:2.2, 5:2.0, 9:4.5, 0:2.3}

stu['sol_type']    = stu['text1'].map(TEXTURE_TO_SOL).fillna('limoneux')
stu['argile_est']  = stu['text1'].map(TEXTURE_TO_ARGILE).fillna(20).astype(float)
stu['ph_est']      = stu['text1'].map(TEXTURE_TO_PH).fillna(6.7).astype(float)
stu['mo_est']      = stu['text1'].map(TEXTURE_TO_MO).fillna(2.3).astype(float)

# STU dominante par SMU
stuorg_dom = stuorg.sort_values('pcarea', ascending=False).drop_duplicates('smu')
smu_props  = stuorg_dom.merge(
    stu[['stu','sol_type','argile_est','ph_est','mo_est','text1']],
    on='stu', how='left'
)

print('=== TABLE SMU → PROPRIÉTÉS SOL ===')
print(f'SMU avec propriétés : {len(smu_props)}')
print()
print('Aperçu :')
print(smu_props[['smu','stu','pcarea','sol_type','argile_est','ph_est','mo_est']]
      .head(8).to_string())
print()
print('Distribution des types de sol (tous polygones France) :')
for s, n in smu_props['sol_type'].value_counts().items():
    print(f'  {s:<20} : {n} SMU ({n/len(smu_props)*100:.1f}%)')
print()
print('⚠️  Rappel : argile_est, ph_est, mo_est sont des ESTIMATIONS')
print('   basées sur des valeurs typiques par type de texture.')
print('   Pas de mesures analytiques réelles à l\'échelle de la parcelle.')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 7 — Extraction des centroïdes Lambert 93        ║
# ║                                                          ║
# ║  On lit le .shp pour extraire les bounding boxes de      ║
# ║  chaque polygone et calculer leur centroïde approximatif ║
# ║  en coordonnées Lambert 93.                              ║
# ╚══════════════════════════════════════════════════════════╝

def read_shp_bboxes(path):
    bboxes = []
    with open(path, 'rb') as f:
        f.seek(100)
        while True:
            header = f.read(8)
            if len(header) < 8: break
            record_num   = struct.unpack('>I', header[:4])[0]
            content_len  = struct.unpack('>I', header[4:])[0] * 2
            content      = f.read(content_len)
            if len(content) < 4: break
            shape_type = struct.unpack('<I', content[:4])[0]
            if shape_type == 5 and len(content) >= 36:
                xmin = struct.unpack('<d', content[4:12])[0]
                ymin = struct.unpack('<d', content[12:20])[0]
                xmax = struct.unpack('<d', content[20:28])[0]
                ymax = struct.unpack('<d', content[28:36])[0]
                bboxes.append({'record':record_num,
                               'cx_l93':(xmin+xmax)/2, 'cy_l93':(ymin+ymax)/2})
    return pd.DataFrame(bboxes)

bboxes = read_shp_bboxes('sols_france/30169_L93.shp')
dbf['record'] = range(1, len(dbf)+1)
poly_sol = bboxes.merge(dbf[['record','SMU']], on='record', how='left')
poly_sol = poly_sol.merge(smu_props.rename(columns={'smu':'SMU'}), on='SMU', how='left')

print('=== CENTROÏDES DES POLYGONES (Lambert 93) ===')
print(f'Polygones avec centroïde    : {len(poly_sol)}')
print(f'Polygones avec sol_type     : {poly_sol.sol_type.notna().sum()}')
print()
print(f'Emprise Lambert 93 :')
print(f'  X : {poly_sol.cx_l93.min():,.0f} m — {poly_sol.cx_l93.max():,.0f} m')
print(f'  Y : {poly_sol.cy_l93.min():,.0f} m — {poly_sol.cy_l93.max():,.0f} m')
print()
print('Zone HdF approximative (Lambert 93) :')
print('  X : 450 000 — 750 000 m')
print('  Y : 6 800 000 — 7 150 000 m')

poly_hdf = poly_sol[
    poly_sol.sol_type.notna() &
    (poly_sol.cx_l93 >= 450000) & (poly_sol.cx_l93 <= 750000) &
    (poly_sol.cy_l93 >= 6800000) & (poly_sol.cy_l93 <= 7150000)
].copy()
print()
print(f'Polygones dans la zone HdF : {len(poly_hdf)}')
print('Distribution sol_type HdF :')
for s, n in poly_hdf.sol_type.value_counts().items():
    print(f'  {s:<20} : {n} polygones ({n/len(poly_hdf)*100:.1f}%)')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 8 — Conversion Lambert 93 → WGS84              ║
# ║                                                          ║
# ║  Les parcelles RPG sont en WGS84 (lat/lon).              ║
# ║  Les polygones SGDBE sont en Lambert 93 (mètres).        ║
# ║  Pour faire la jointure spatiale, on convertit           ║
# ║  les coordonnées des parcelles RPG en Lambert 93.        ║
# ╚══════════════════════════════════════════════════════════╝

def wgs84_to_l93(lon_deg, lat_deg):
    """Conversion WGS84 → Lambert 93. Précision ~1m."""
    lon = math.radians(lon_deg)
    lat = math.radians(lat_deg)
    lon0 = math.radians(3.0)
    phi1 = math.radians(44.0)
    phi2 = math.radians(49.0)
    phi0 = math.radians(46.5)
    x0, y0 = 700000.0, 6600000.0
    a = 6378137.0
    e2 = 0.00669437999014
    e = math.sqrt(e2)

    def alpha(phi):
        s = e * math.sin(phi)
        return math.tan(math.pi/4 + phi/2) * ((1-s)/(1+s))**(e/2)

    n = (math.log(math.cos(phi1)/math.sqrt(1-e2*math.sin(phi1)**2)) -
         math.log(math.cos(phi2)/math.sqrt(1-e2*math.sin(phi2)**2))) / \
        (math.log(alpha(phi2)) - math.log(alpha(phi1)))
    c  = (math.cos(phi1)/math.sqrt(1-e2*math.sin(phi1)**2)) * alpha(phi1)**n / n
    r0 = a * c * alpha(phi0)**(-n)
    r  = a * c * alpha(lat)**(-n)
    theta = n * (lon - lon0)
    return x0 + r*math.sin(theta), y0 + r0 - r*math.cos(theta)

# Charger les coordonnées RPG
coords = pd.read_csv('rpg_coords.csv')
print(f'Parcelles RPG à convertir : {len(coords):,}')
print()

# Conversion WGS84 → Lambert 93
print('Conversion WGS84 → Lambert 93...')
xs, ys = [], []
for _, row in coords.iterrows():
    try:
        x, y = wgs84_to_l93(row.lon, row.lat)
        xs.append(x); ys.append(y)
    except:
        xs.append(None); ys.append(None)

coords['x_l93'] = xs
coords['y_l93'] = ys
coords_ok = coords.dropna(subset=['x_l93','y_l93'])

print(f'✓ {len(coords_ok):,} parcelles converties')
print()
print('Vérification — emprise HdF en Lambert 93 :')
print(f'  X : {coords_ok.x_l93.min():,.0f} — {coords_ok.x_l93.max():,.0f} m')
print(f'  Y : {coords_ok.y_l93.min():,.0f} — {coords_ok.y_l93.max():,.0f} m')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 9 — Jointure spatiale KDTree                    ║
# ║                                                          ║
# ║  On utilise un KDTree (algorithme du plus proche voisin) ║
# ║  pour assigner à chaque parcelle RPG le polygone de sol  ║
# ║  SGDBE le plus proche.                                   ║
# ║                                                          ║
# ║  KDTree : structure de données spatiale qui permet de    ║
# ║  trouver le voisin le plus proche en O(log n) au lieu    ║
# ║  de O(n) pour une recherche naïve.                       ║
# ║  → Essentiel pour 566 000 parcelles × 530 polygones.     ║
# ╚══════════════════════════════════════════════════════════╝

from scipy.spatial import cKDTree

print('Jointure spatiale KDTree (parcelles RPG → polygones sol)...')

# Construire le KDTree sur les centroïdes des polygones HdF
tree = cKDTree(poly_hdf[['cx_l93','cy_l93']].values)

# Requête : pour chaque parcelle, trouver le polygone le plus proche
parc_xy = coords_ok[['x_l93','y_l93']].values
distances, indices = tree.query(parc_xy, k=1)

# Assignation des propriétés sol
sol_cols = poly_hdf[['sol_type','argile_est','ph_est','mo_est']].iloc[indices].reset_index(drop=True)
result = pd.DataFrame({
    'id_parcel':     coords_ok['id_parcel'].values,
    'sol_type':      sol_cols['sol_type'].values,
    'argile_pct_sol':sol_cols['argile_est'].values,
    'ph_sol_reel':   sol_cols['ph_est'].values,
    'mo_sol_reel':   sol_cols['mo_est'].values,
    'dist_sol_km':   (distances / 1000).round(1),
})

print(f'✓ {len(result):,} parcelles avec type de sol assigné')
print()
print('=== QUALITÉ DE LA JOINTURE ===')
print(f'  Distance moyenne au polygone : {result.dist_sol_km.mean():.1f} km')
print(f'  Distance médiane             : {result.dist_sol_km.median():.1f} km')
print(f'  Distance max                 : {result.dist_sol_km.max():.1f} km')
print(f'  Parcelles à < 5 km           : {(result.dist_sol_km < 5).sum():,} '
      f'({(result.dist_sol_km < 5).mean()*100:.1f}%)')
print(f'  Parcelles à < 10 km          : {(result.dist_sol_km < 10).sum():,} '
      f'({(result.dist_sol_km < 10).mean()*100:.1f}%)')
print()
print('Limite de la résolution 1/1 000 000 :')
print('  La distance médiane de 5.8 km reflète la résolution grossière')
print('  du SGDBE — un polygone couvre ~25 km² en HdF.')
print('  → Sol_type réel, mais argile/pH/MO estimés (pas mesurés).')

result.to_csv('sol_par_parcelle.csv', index=False)
print('\n✓ sol_par_parcelle.csv sauvegardé')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 10 — Analyse du sol assigné aux parcelles HdF   ║
# ╚══════════════════════════════════════════════════════════╝

print('=== DISTRIBUTION DES TYPES DE SOL — PARCELLES RPG HdF ===')
print()
dist_sol = result['sol_type'].value_counts()
print(f'{"Type de sol":<22} {"N parcelles":>12} {"% total":>8}')
print('-'*46)
for s, n in dist_sol.items():
    print(f'  {s:<20} {n:>12,} {n/len(result)*100:>7.1f}%')
print()

print('=== STATISTIQUES ARGILE / pH / MO PAR TYPE DE SOL ===')
print(f'{"Type":<20} {"Argile moy":>11} {"pH moy":>8} {"MO moy":>8}  Signification')
print('-'*72)
SIGNIF = {
    'limoneux':         'Sol dominant HdF — très fertile, bon drainage',
    'argilo_limoneux':  'Sol lourd — bonne rétention eau, labour difficile',
    'sablo_limoneux':   'Sol léger — bon ressuyage, idéal pdt',
    'argileux_lourd':   'Sol très lourd — risque engorgement',
    'tourbe':           'Sol organique — rare HdF, zones humides',
    'craie':            'Sol calcaire — pH élevé, réserve eau limitée',
}
for s in dist_sol.index:
    sub = result[result.sol_type == s]
    sig = SIGNIF.get(s, '—')
    print(f'  {s:<18} {sub.argile_pct_sol.mean():>9.1f}%  '
          f'{sub.ph_sol_reel.mean():>7.2f}  '
          f'{sub.mo_sol_reel.mean():>7.2f}%  {sig}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 11 — Visualisations                             ║
# ╚══════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

types_ord = dist_sol.index.tolist()
cols = [COULEURS_SOL.get(t, '#888') for t in types_ord]

# ── 1. Distribution types de sol ──
ax = axes[0]
bars = ax.barh(types_ord, dist_sol.values, color=cols, edgecolor='white')
ax.set_xlabel('Nombre de parcelles')
ax.set_title('Types de sol assignés\naux parcelles RPG HdF', fontsize=12, pad=10)
for bar, n in zip(bars, dist_sol.values):
    ax.text(bar.get_width()+500, bar.get_y()+bar.get_height()/2,
            f'{n/len(result)*100:.1f}%', va='center', fontsize=9)
ax.set_xlim(0, dist_sol.max()*1.18)
ax.invert_yaxis()

# ── 2. Distribution des distances ──
ax = axes[1]
ax.hist(result['dist_sol_km'], bins=50, color='#3b82f6', edgecolor='white', alpha=0.8)
ax.axvline(result.dist_sol_km.median(), color='red', linestyle='--',
           linewidth=1.5, label=f'Médiane : {result.dist_sol_km.median():.1f} km')
ax.axvline(result.dist_sol_km.mean(), color='orange', linestyle='--',
           linewidth=1.5, label=f'Moyenne : {result.dist_sol_km.mean():.1f} km')
ax.set_xlabel('Distance au polygone de sol (km)')
ax.set_ylabel('Nombre de parcelles')
ax.set_title('Distribution des distances\nde jointure spatiale', fontsize=12, pad=10)
ax.legend(fontsize=9)

# ── 3. % argile par type de sol ──
ax = axes[2]
argiles = [result[result.sol_type==t]['argile_pct_sol'].mean() for t in types_ord]
bars = ax.bar(range(len(types_ord)), argiles, color=cols, edgecolor='white')
ax.set_xticks(range(len(types_ord)))
ax.set_xticklabels([t.replace('_','-') for t in types_ord],
                   rotation=30, ha='right', fontsize=9)
ax.set_ylabel('% argile estimé')
ax.set_title('Teneur en argile estimée\npar type de sol', fontsize=12, pad=10)
for bar, a in zip(bars, argiles):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{a:.0f}%', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Analyse SGDBE INRAE — Sol des parcelles HdF', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('sgdbe_analyse_sol.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ sgdbe_analyse_sol.png sauvegardé')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 12 — Synthèse et téléchargements                ║
# ╚══════════════════════════════════════════════════════════╝

print('=' * 60)
print('  SYNTHÈSE — SGDBE INRAE')
print('=' * 60)
print()
print('  Source         : data.gouv.fr — référence 30169')
print('  Résolution     : 1/1 000 000')
print(f'  Polygones France : {len(poly_sol):,}')
print(f'  Polygones HdF    : {len(poly_hdf)}')
print(f'  Parcelles RPG assignées : {len(result):,} (100%)')
print()
print('  Ce qui est RÉEL :')
print('    sol_type → type de sol officiel INRAE (limoneux, argileux...)')
print()
print('  Ce qui est ESTIMÉ (valeurs typiques par texture) :')
print('    argile_pct → % argile estimé depuis le code texture')
print('    ph_sol     → pH estimé depuis le code texture')
print('    mo_pct     → % matières organiques estimé')
print()
print('  Distance médiane de jointure : 5.8 km')
print('  (limite inhérente à la résolution 1/1 000 000)')
print()
print('  Fichiers produits :')
print('    sol_par_parcelle.csv   → id_parcel + sol_type + argile + pH + MO')
print('    sgdbe_analyse_sol.png  → visualisations')

files.download('sol_par_parcelle.csv')
files.download('sgdbe_analyse_sol.png')
print('\n✓ Téléchargements lancés')